# Handcrafted Image Retrieval Pipeline
## Oxford & Paris Buildings Datasets

### Pipeline Overview

| Stage | Component | Source |
|-------|-----------|--------|
| 🔴 Offline | **RootSIFT** extraction | Arandjelović & Zisserman (2012) |
| 🔴 Offline | **VLAD** encoding | Jégou et al. (CVPR 2010) |
| 🔴 Offline | **PCA + Whitening** | Jégou & Chum (ECCV 2012) |
| 🟢 Online  | Cosine similarity retrieval | Philbin et al. (CVPR 2007) |
| 🟢 Online  | **RANSAC** spatial re-ranking | Philbin et al. (CVPR 2007) |
| 🟢 Online  | **AQE** query expansion | Chum et al. (ICCV 2007) |

### Dataset structure (Kaggle)
```
/kaggle/input/datasets/jeffreyamc/oxford-paris-buildings-v2/
    oxford/
        all_souls/   ← *.jpg images + *_query.txt, *_good.txt, *_ok.txt, *_junk.txt
        ashmolean/
        balliol/
        bodleian/ ...
    paris/
        defense/
        eiffel/ ...
```

---
## Cell 1 — Imports & Setup

In [ ]:
import os, sys, time, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

print(f'OpenCV : {cv2.__version__}')
print(f'NumPy  : {np.__version__}')

import subprocess
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
    print(f'GPU    : {r.stdout.strip()}')
except:
    print('GPU info not available')

---
## Cell 2 — Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  PATHS
# ─────────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path('/kaggle/input/datasets/jeffreyamc/oxford-paris-buildings-v2')
OXFORD_ROOT  = DATASET_ROOT / 'oxford'
PARIS_ROOT   = DATASET_ROOT / 'paris'

# Select which datasets to index: any combination of ['oxford', 'paris']
DATASETS_TO_USE = ['oxford', 'paris']

OUTPUT_DIR = Path('/kaggle/working/retrieval_output')
CACHE_DIR  = OUTPUT_DIR / 'cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────
#  HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────
K_VLAD          = 256    # VLAD visual vocabulary size
PCA_DIM         = 256    # Dimension after PCA + whitening
SIFT_N_FEATURES = 2000   # Max SIFT keypoints per image
MAX_IMAGE_SIZE  = 800    # Resize longest side to this before SIFT
N_WORKERS       = 4      # Parallel threads for feature extraction
RERANK_TOP_N    = 100    # Candidates to spatially re-rank with RANSAC
MIN_INLIERS     = 6      # RANSAC inlier threshold
AQE_TOP_K       = 10     # Images used in query expansion

print('Configuration OK')
print(f'  Datasets    : {DATASETS_TO_USE}')
print(f'  K_VLAD      : {K_VLAD}  |  PCA_DIM : {PCA_DIM}')
print(f'  RERANK_TOP_N: {RERANK_TOP_N}  |  AQE_TOP_K: {AQE_TOP_K}')

---
## Cell 3 — Dataset Scanner
Reads the per-landmark sub-folders, collects all image paths, and parses the ground-truth files.

In [ ]:
def scan_dataset(dataset_root: Path, dataset_name: str):
    """
    Scan  <dataset_root>/<landmark_name>/  folders.
    Each landmark folder contains:
        *.jpg                              — images
        <name>_query.txt                   — 'oxc1_STEM x1 y1 x2 y2'
        <name>_good.txt / _ok.txt / _junk.txt

    Returns
    -------
    img_paths : list[Path]
    stem2path : dict[str, Path]
    queries   : list[dict]
    """
    img_paths, stem2path, queries = [], {}, []

    if not dataset_root.exists():
        print(f'[WARN] {dataset_root} not found — skipping {dataset_name}')
        return img_paths, stem2path, queries

    landmark_dirs = sorted(d for d in dataset_root.iterdir() if d.is_dir())
    print(f'\n{dataset_name.upper()} — {len(landmark_dirs)} landmark folders')

    # Helper: strip known prefixes
    def clean(s):
        for pfx in ('oxc1_', 'paris_', 'oxf_'):
            s = s.replace(pfx, '')
        return s

    for lm_dir in landmark_dirs:
        # Collect images
        imgs = (sorted(lm_dir.glob('*.jpg')) +
                sorted(lm_dir.glob('*.JPG')) +
                sorted(lm_dir.glob('*.png')))
        for p in imgs:
            img_paths.append(p)
            stem2path[p.stem] = p

        # Parse ground truth
        for qf in sorted(lm_dir.glob('*_query.txt')):
            q_name = qf.stem.replace('_query', '')

            with open(qf) as f:
                parts = f.read().strip().split()

            q_img = clean(parts[0])
            q_roi = list(map(float, parts[1:5])) if len(parts) >= 5 else None

            def read_set(suffix, lm=lm_dir, qn=q_name):
                p = lm / f'{qn}_{suffix}.txt'
                if not p.exists():
                    return set()
                with open(p) as f:
                    return {clean(l.strip()) for l in f if l.strip()}

            queries.append({
                'name'      : f'{dataset_name}_{q_name}',
                'landmark'  : lm_dir.name,
                'dataset'   : dataset_name,
                'query_img' : q_img,
                'query_roi' : q_roi,
                'good'      : read_set('good'),
                'ok'        : read_set('ok'),
                'junk'      : read_set('junk'),
            })

        n_q = sum(1 for q in queries if q['landmark'] == lm_dir.name)
        print(f'  {lm_dir.name:20s}: {len(imgs):4d} images, {n_q} queries')

    print(f'  => TOTAL: {len(img_paths)} images, {len(queries)} queries')
    return img_paths, stem2path, queries


# ── Scan ──────────────────────────────────────────────────────────────────
ALL_IMG_PATHS = []
STEM2PATH     = {}
ALL_QUERIES   = []

if 'oxford' in DATASETS_TO_USE:
    ox_imgs, ox_s2p, ox_q = scan_dataset(OXFORD_ROOT, 'oxford')
    ALL_IMG_PATHS += ox_imgs
    STEM2PATH.update(ox_s2p)
    ALL_QUERIES   += ox_q

if 'paris' in DATASETS_TO_USE:
    pa_imgs, pa_s2p, pa_q = scan_dataset(PARIS_ROOT, 'paris')
    ALL_IMG_PATHS += pa_imgs
    STEM2PATH.update(pa_s2p)
    ALL_QUERIES   += pa_q

N_IMAGES = len(ALL_IMG_PATHS)
# Ordered stem list — row i of the DB matrix corresponds to DB_STEMS[i]
DB_STEMS    = [p.stem for p in ALL_IMG_PATHS]
STEM_TO_IDX = {s: i for i, s in enumerate(DB_STEMS)}

print(f'\n{"="*50}')
print(f' DATABASE SIZE : {N_IMAGES} images')
print(f' TOTAL QUERIES : {len(ALL_QUERIES)}')
print(f'{"="*50}')

---
# 🔴 OFFLINE STAGE
Run **once** and cache to disk.  The outputs are:
- `vocab_K*.pkl`   — visual vocabulary (K-Means centroids)
- `vlad_raw_K*.npy` — raw VLAD vectors for every DB image
- `pca_K*_D*.pkl`  — fitted PCA+whitening model
- `db_index_*.npy` — final projected + normalized DB matrix

## Offline Step 1 — RootSIFT Extractor
> **Arandjelović & Zisserman (2012)**  
> L1-normalize SIFT → element-wise √.  
> Euclidean distance on RootSIFT ≡ Hellinger kernel on SIFT.

In [ ]:
def compute_rootsift(descs: np.ndarray) -> np.ndarray:
    """Convert SIFT → RootSIFT (Arandjelović & Zisserman, 2012)."""
    descs = descs.astype(np.float32)
    descs /= (descs.sum(axis=1, keepdims=True) + 1e-7)  # L1 normalize
    descs  = np.sqrt(descs)                              # element-wise sqrt
    return descs  # already L2-normalized


def extract_rootsift(image_path: Path,
                     roi=None,
                     max_size: int = MAX_IMAGE_SIZE,
                     n_features: int = SIFT_N_FEATURES):
    """
    Detect keypoints + compute RootSIFT descriptors.
    roi=[x1,y1,x2,y2]: keep only keypoints inside the bounding box
                        (used for query images with annotated ROI).
    Returns (keypoints, descriptors) or ([], None) on failure.
    """
    try:
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return [], None

        h, w  = img.shape
        scale = min(max_size / max(h, w), 1.0)
        if scale < 1.0:
            img = cv2.resize(img, (int(w*scale), int(h*scale)),
                             interpolation=cv2.INTER_AREA)

        sift = cv2.SIFT_create(nfeatures=n_features,
                               contrastThreshold=0.04,
                               edgeThreshold=10)
        kps, descs = sift.detectAndCompute(img, None)
        if descs is None or len(descs) == 0:
            return [], None

        descs = compute_rootsift(descs)

        if roi is not None:
            x1, y1, x2, y2 = [c*scale for c in roi]
            mask  = np.array([x1 <= kp.pt[0] <= x2 and y1 <= kp.pt[1] <= y2
                              for kp in kps], dtype=bool)
            kps   = [kp for kp, m in zip(kps, mask) if m]
            descs = descs[mask] if mask.any() else None
            if descs is None or len(descs) == 0:
                return [], None

        return kps, descs
    except Exception as e:
        print(f'  [ERR] {image_path.name}: {e}')
        return [], None


# Sanity check
if ALL_IMG_PATHS:
    kps, descs = extract_rootsift(ALL_IMG_PATHS[0])
    if descs is not None:
        print(f'RootSIFT OK — {ALL_IMG_PATHS[0].name}: {len(kps)} kps, '
              f'shape={descs.shape}, norm={np.linalg.norm(descs[0]):.4f}')

## Offline Step 2 — Build Visual Vocabulary (K-Means)

In [ ]:
VOCAB_CACHE = CACHE_DIR / f'vocab_K{K_VLAD}.pkl'

def build_vocabulary(image_paths, k=K_VLAD, max_desc_per_img=150,
                     n_sample_imgs=2000, cache_path=None):
    """
    Fit MiniBatchKMeans on a sample of RootSIFT descriptors.
    Cached after first run.
    """
    if cache_path and cache_path.exists():
        print(f'[CACHE] Vocabulary loaded from {cache_path.name}')
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f'Building K={k} vocabulary on '
          f'{min(n_sample_imgs, len(image_paths))} images...')
    t0 = time.time()

    rng  = np.random.default_rng(42)
    idxs = rng.choice(len(image_paths), min(n_sample_imgs, len(image_paths)),
                      replace=False)
    all_descs = []
    for i in tqdm(idxs, desc='Collecting descriptors'):
        _, d = extract_rootsift(image_paths[i])
        if d is not None and len(d) > 0:
            if len(d) > max_desc_per_img:
                sel = np.random.choice(len(d), max_desc_per_img, replace=False)
                d = d[sel]
            all_descs.append(d)

    if not all_descs:
        raise RuntimeError('No descriptors found — check image paths!')

    X = np.vstack(all_descs).astype(np.float32)
    print(f'  Clustering {len(X):,} descriptors → {k} words...')

    km = MiniBatchKMeans(n_clusters=k, random_state=42,
                         batch_size=min(20_000, len(X)),
                         max_iter=300, n_init=5, verbose=0)
    km.fit(X)
    print(f'  Done in {time.time()-t0:.1f}s')

    if cache_path:
        with open(cache_path, 'wb') as f:
            pickle.dump(km, f)
        print(f'  Cached → {cache_path.name}')
    return km


VOCABULARY = build_vocabulary(ALL_IMG_PATHS, k=K_VLAD, cache_path=VOCAB_CACHE)
CENTROIDS  = VOCABULARY.cluster_centers_.astype(np.float32)  # (K, 128)
print(f'Centroids shape: {CENTROIDS.shape}')

## Offline Step 3 — VLAD Encoding
> **Jégou et al. (CVPR 2010)**  
> $v_{i,j} = \sum_{x:\,NN(x)=c_i} (x_j - c_{i,j})$  
> Power normalization (SSR) + L2 normalize.

In [ ]:
def vlad_encode(descs: np.ndarray, centroids: np.ndarray,
                power_norm: bool = True) -> np.ndarray:
    """
    Encode descriptors as a VLAD vector.

    1. Hard-assign each descriptor to its nearest centroid.
    2. Accumulate residuals per centroid.
    3. [Optional] Signed square-root (power norm) — reduces burstiness
       (Jégou & Chum, ECCV 2012).
    4. L2 normalize.

    Returns (K*128,) vector.
    """
    K, D = centroids.shape

    # Fast assignment: ||x-c||^2 = ||x||^2 - 2x·c + ||c||^2
    x_sq   = (descs**2).sum(1, keepdims=True)        # (N,1)
    c_sq   = (centroids**2).sum(1, keepdims=True).T   # (1,K)
    dot    = descs @ centroids.T                       # (N,K)
    assign = np.argmin(x_sq + c_sq - 2.0*dot, axis=1) # (N,)

    # Vectorised residual accumulation
    vlad = np.zeros((K, D), dtype=np.float32)
    np.add.at(vlad, assign, descs - centroids[assign])

    if power_norm:
        vlad = np.sign(vlad) * np.sqrt(np.abs(vlad))

    vlad = vlad.flatten()
    n    = np.linalg.norm(vlad)
    if n > 1e-8:
        vlad /= n
    return vlad


def extract_vlad(image_path: Path, centroids: np.ndarray, roi=None) -> np.ndarray:
    """Convenience: RootSIFT extraction + VLAD encoding for one image."""
    _, descs = extract_rootsift(image_path, roi=roi)
    if descs is None or len(descs) == 0:
        return np.zeros(len(centroids)*128, dtype=np.float32)
    return vlad_encode(descs, centroids)


if ALL_IMG_PATHS:
    v = extract_vlad(ALL_IMG_PATHS[0], CENTROIDS)
    print(f'VLAD shape: {v.shape}  norm={np.linalg.norm(v):.4f}')

## Offline Step 4 — Extract VLAD for ALL Database Images

In [ ]:
VLAD_RAW_CACHE  = CACHE_DIR / f'vlad_raw_K{K_VLAD}.npy'
DB_STEMS_CACHE  = CACHE_DIR / 'db_stems.pkl'

def build_database(image_paths, centroids,
                   cache_vlad=None, cache_stems=None):
    """
    OFFLINE: Extract and store VLAD vectors for every image in the database.
    Returns (vlad_matrix[N, K*128], stems[N]).
    """
    if cache_vlad and cache_vlad.exists() and \
       cache_stems and cache_stems.exists():
        print(f'[CACHE] DB loaded from {cache_vlad.name}')
        vlad_matrix = np.load(cache_vlad)
        with open(cache_stems, 'rb') as f:
            stems = pickle.load(f)
        print(f'  {len(stems)} vectors, dim={vlad_matrix.shape[1]}')
        return vlad_matrix, stems

    print(f'[OFFLINE] Extracting VLAD for {len(image_paths)} images...')
    t0          = time.time()
    vlad_dim    = len(centroids) * 128
    vlad_matrix = np.zeros((len(image_paths), vlad_dim), dtype=np.float32)
    stems_out   = [''] * len(image_paths)
    n_failed    = 0

    def worker(args):
        idx, path = args
        return idx, path.stem, extract_vlad(path, centroids)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(worker, (i, p)): i
                for i, p in enumerate(image_paths)}
        for fut in tqdm(as_completed(futs), total=len(futs),
                        desc='VLAD extraction'):
            try:
                idx, stem, v = fut.result()
                vlad_matrix[idx] = v
                stems_out[idx]   = stem
            except Exception:
                n_failed += 1

    print(f'  Done in {time.time()-t0:.1f}s  ({n_failed} failures)')

    if cache_vlad:
        np.save(cache_vlad, vlad_matrix)
        with open(cache_stems, 'wb') as f:
            pickle.dump(stems_out, f)
        print(f'  Cached → {cache_vlad.name}')

    return vlad_matrix, stems_out


VLAD_RAW, DB_STEMS = build_database(
    ALL_IMG_PATHS, CENTROIDS,
    cache_vlad=VLAD_RAW_CACHE,
    cache_stems=DB_STEMS_CACHE)

STEM_TO_IDX = {s: i for i, s in enumerate(DB_STEMS)}
print(f'\nVLAD_RAW shape: {VLAD_RAW.shape}')

## Offline Step 5 — PCA + Whitening
> **Jégou & Chum (ECCV 2012)**  
> $\hat{X} = \mathrm{diag}(\lambda_1^{-1/2},\dots,\lambda_{D_0}^{-1/2})\,P^\top X$  then L2 re-normalize.  
> - Mean subtraction → handles **co-missing visual words** (negative evidence)  
> - Eigenvalue scaling → handles **co-occurring visual words** (burstiness)

In [ ]:
PCA_CACHE  = CACHE_DIR / f'pca_K{K_VLAD}_D{PCA_DIM}.pkl'
DBIDX_PATH = CACHE_DIR / f'db_index_K{K_VLAD}_D{PCA_DIM}.npy'

def fit_pca_whitening(vlad_matrix, n_components=PCA_DIM, cache_path=None):
    """
    OFFLINE: Fit PCA(whiten=True) on the raw VLAD matrix.
    sklearn PCA(whiten=True) applies exactly:
        diag(lambda^{-1/2}) P^T (x - mean)   [Jégou & Chum 2012, Eq. 5]
    """
    if cache_path and cache_path.exists():
        print(f'[CACHE] PCA loaded from {cache_path.name}')
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f'[OFFLINE] Fitting PCA (n={n_components}, whiten=True)...')
    t0  = time.time()
    pca = PCA(n_components=n_components, whiten=True, random_state=42)
    pca.fit(vlad_matrix)
    print(f'  Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%'
          f'  ({time.time()-t0:.1f}s)')

    if cache_path:
        with open(cache_path, 'wb') as f:
            pickle.dump(pca, f)
        print(f'  Cached → {cache_path.name}')
    return pca


def apply_pca_whitening(vlad_matrix: np.ndarray, pca) -> np.ndarray:
    """
    Project + whiten + **L2 re-normalize** (the critical step from Jégou & Chum).
    """
    proj = pca.transform(vlad_matrix)
    proj = normalize(proj, norm='l2')
    return proj.astype(np.float32)


PCA_MODEL = fit_pca_whitening(VLAD_RAW, n_components=PCA_DIM, cache_path=PCA_CACHE)

# Project all DB vectors
if DBIDX_PATH.exists():
    print(f'[CACHE] Loading DB index from {DBIDX_PATH.name}')
    DB_VLAD = np.load(DBIDX_PATH)
else:
    print('[OFFLINE] Projecting DB vectors...')
    DB_VLAD = apply_pca_whitening(VLAD_RAW, PCA_MODEL)
    np.save(DBIDX_PATH, DB_VLAD)
    print(f'  Cached → {DBIDX_PATH.name}')

norms = np.linalg.norm(DB_VLAD, axis=1)
print(f'DB index shape : {DB_VLAD.shape}')
print(f'Norm stats     : mean={norms.mean():.4f}, std={norms.std():.4f}')
print('\n✅  OFFLINE STAGE COMPLETE')

---
# 🟢 ONLINE STAGE
Runs at query time. Uses the pre-built DB index.

## Online Step 1 — Query Encoding

In [ ]:
def encode_query(image_path: Path, roi=None) -> np.ndarray:
    """
    ONLINE: RootSIFT → VLAD → PCA+whitening → L2 normalize.
    Uses the same vocabulary and PCA model built offline.
    roi=[x1,y1,x2,y2] restricts features to the annotated region.
    Returns (PCA_DIM,) vector.
    """
    _, descs = extract_rootsift(image_path, roi=roi)
    if descs is None or len(descs) == 0:
        return np.zeros(PCA_DIM, dtype=np.float32)
    vlad = vlad_encode(descs, CENTROIDS)
    return apply_pca_whitening(vlad[np.newaxis], PCA_MODEL)[0]


print('Query encoder ready.')

## Online Step 2 — Initial Retrieval (Cosine Similarity)

In [ ]:
def initial_retrieve(query_vec: np.ndarray,
                     db_matrix: np.ndarray,
                     exclude: str = None):
    """
    ONLINE: Rank all DB images by cosine similarity to the query.
    Vectors are L2-normalised → cosine similarity = dot product.
    Returns (ranked_stems, scores).
    """
    sims       = db_matrix @ query_vec
    ranked_idx = np.argsort(-sims)
    stems      = [DB_STEMS[i] for i in ranked_idx]
    scores     = sims[ranked_idx]
    if exclude:
        mask   = [s != exclude for s in stems]
        stems  = [s for s, m in zip(stems, mask) if m]
        scores = scores[np.array(mask)]
    return stems, scores


print('Initial retrieval ready.')

## Online Step 3 — RANSAC Spatial Re-ranking
> **Philbin et al. (CVPR 2007)**  
> FLANN match → Lowe ratio test → RANSAC homography → count inliers.

In [ ]:
def ransac_rerank(query_path: Path, query_roi,
                  ranked_stems: list,
                  top_n: int = RERANK_TOP_N,
                  min_inliers: int = MIN_INLIERS) -> list:
    """
    ONLINE: Re-rank top-N candidates with RANSAC homography verification.

    For each candidate:
        1. Extract RootSIFT from both query (inside ROI) and candidate.
        2. FLANN k-NN matching (k=2) + Lowe ratio test (0.75).
        3. RANSAC homography → count inliers.
    Candidates with >= min_inliers inliers are ranked first.
    """
    to_rerank = ranked_stems[:top_n]
    rest      = ranked_stems[top_n:]

    q_kps, q_descs = extract_rootsift(query_path, roi=query_roi)
    if q_descs is None or len(q_kps) < 4:
        return ranked_stems

    q_pts = np.float32([kp.pt for kp in q_kps])
    flann = cv2.FlannBasedMatcher(
        dict(algorithm=1, trees=5),
        dict(checks=50))

    inliers = {}
    for stem in to_rerank:
        path = STEM2PATH.get(stem)
        if path is None:
            inliers[stem] = 0; continue

        c_kps, c_descs = extract_rootsift(path)
        if c_descs is None or len(c_kps) < 4:
            inliers[stem] = 0; continue

        c_pts = np.float32([kp.pt for kp in c_kps])
        try:
            matches = flann.knnMatch(q_descs, c_descs, k=2)
            good    = [m for pair in matches if len(pair)==2
                       for m, n in [pair] if m.distance < 0.75*n.distance]
            if len(good) < 4:
                inliers[stem] = 0; continue

            src = np.float32([q_pts[m.queryIdx] for m in good]).reshape(-1,1,2)
            dst = np.float32([c_pts[m.trainIdx] for m in good]).reshape(-1,1,2)
            _, mask = cv2.findHomography(src, dst, cv2.RANSAC,
                                         ransacReprojThreshold=10.0)
            inliers[stem] = int(mask.sum()) if mask is not None else 0
        except Exception:
            inliers[stem] = 0

    def key(s):
        c = inliers.get(s, 0)
        return (0 if c >= min_inliers else 1, -c)

    return sorted(to_rerank, key=key) + list(rest)


print('RANSAC re-ranker ready.')

## Online Step 4 — Average Query Expansion (AQE)
> **Chum et al. (ICCV 2007)**  
> $q_{\text{exp}} = \text{normalize}\bigl(\text{mean}(q, v_1, \dots, v_k)\bigr)$  then re-retrieve.

In [ ]:
def average_query_expansion(query_vec: np.ndarray,
                             ranked_stems: list,
                             top_k: int = AQE_TOP_K) -> np.ndarray:
    """
    ONLINE: Build enriched query by averaging with top-k DB vectors,
    then L2-renormalize (Chum et al. ICCV 2007).
    """
    vecs = [query_vec]
    for stem in ranked_stems[:top_k]:
        idx = STEM_TO_IDX.get(stem)
        if idx is not None:
            vecs.append(DB_VLAD[idx])
    if len(vecs) == 1:
        return query_vec
    exp = np.mean(vecs, axis=0).astype(np.float32)
    n   = np.linalg.norm(exp)
    return exp / n if n > 1e-8 else exp


print('AQE module ready.')

## Online — Full Pipeline (Single Query)

In [ ]:
def run_query(query_info: dict,
              use_ransac: bool = True,
              use_aqe: bool    = True) -> tuple:
    """
    ONLINE full pipeline for one query dict.

    Flow:
        encode_query
            ↓
        initial_retrieve  (cosine sim)
            ↓  [optional]
        ransac_rerank
            ↓  [optional]
        average_query_expansion  →  initial_retrieve  (2nd pass)

    Returns (ranked_stems, ap, precision@5).
    """
    q_stem = query_info['query_img']
    q_path = STEM2PATH.get(q_stem)
    if q_path is None:
        return [], 0.0, 0.0

    q_roi = query_info['query_roi']
    good  = query_info['good']
    ok    = query_info['ok']
    junk  = query_info['junk']

    # 1. Encode
    q_vec = encode_query(q_path, roi=q_roi)

    # 2. Initial retrieve
    ranked, scores = initial_retrieve(q_vec, DB_VLAD, exclude=q_stem)

    # 3. RANSAC spatial re-ranking
    if use_ransac:
        ranked = ransac_rerank(q_path, q_roi, ranked)

    # 4. AQE → second pass
    if use_aqe:
        exp_q  = average_query_expansion(q_vec, ranked)
        ranked, _ = initial_retrieve(exp_q, DB_VLAD, exclude=q_stem)

    ap = _ap(ranked, good, ok, junk)
    p5 = _pk(ranked, good, ok, junk, k=5)
    return ranked, ap, p5


# ── Metric helpers ────────────────────────────────────────────────────────
def _ap(ranked, good, ok, junk):
    pos = good | ok
    if not pos: return 0.0
    ap = n_ret = n_rel = 0
    for s in ranked:
        if s in junk: continue
        n_ret += 1
        if s in pos:
            n_rel += 1
            ap    += n_rel / n_ret
    return ap / len(pos)

def _pk(ranked, good, ok, junk, k=5):
    pos = good | ok
    n_seen = n_rel = 0
    for s in ranked:
        if s in junk: continue
        n_seen += 1
        if s in pos: n_rel += 1
        if n_seen == k: break
    return n_rel/k if k else 0.0


print('Full online pipeline ready.')

---
# 📊 EVALUATION — mAP over all queries

In [ ]:
def evaluate(queries, use_ransac=True, use_aqe=True, tag=''):
    all_ap, all_p5, results = [], [], {}
    for q in tqdm(queries, desc=tag or 'Eval'):
        ranked, ap, p5 = run_query(q, use_ransac=use_ransac, use_aqe=use_aqe)
        all_ap.append(ap); all_p5.append(p5)
        results[q['name']] = {'ranked': ranked, 'ap': ap, 'p5': p5,
                               'query_info': q}
    mAP = float(np.mean(all_ap))
    mP5 = float(np.mean(all_p5))
    print(f'  {tag:40s} mAP={mAP:.4f}  P@5={mP5:.4f}')
    return mAP, mP5, results


oxford_queries = [q for q in ALL_QUERIES if q['dataset']=='oxford']
paris_queries  = [q for q in ALL_QUERIES if q['dataset']=='paris']
print(f'Oxford: {len(oxford_queries)} queries  |  Paris: {len(paris_queries)} queries')

In [ ]:
# ── Baseline ─────────────────────────────────────────────────────────────
print('='*55, '\nBASELINE (no RANSAC, no AQE)\n' + '='*55)
mAP_ox_b = mAP_pa_b = 0.0
res_ox_b = res_pa_b = {}

if oxford_queries:
    mAP_ox_b, _, res_ox_b = evaluate(oxford_queries, False, False, 'Oxford-Baseline')
if paris_queries:
    mAP_pa_b, _, res_pa_b = evaluate(paris_queries, False, False, 'Paris-Baseline')

In [ ]:
# ── Full Pipeline ─────────────────────────────────────────────────────────
print('='*55, '\nFULL PIPELINE (RANSAC + AQE)\n' + '='*55)
mAP_ox_f = mAP_pa_f = 0.0
res_ox_f = res_pa_f = {}

if oxford_queries:
    mAP_ox_f, _, res_ox_f = evaluate(oxford_queries, True, True, 'Oxford-Full')
if paris_queries:
    mAP_pa_f, _, res_pa_f = evaluate(paris_queries, True, True, 'Paris-Full')

---
# 📊 ABLATION STUDY

In [ ]:
ablation = {}
eval_qs  = oxford_queries or paris_queries
eval_tag = 'Oxford' if oxford_queries else 'Paris'

for label, ransac, aqe in [
    ('Baseline (VLAD+PCA-Wh)', False, False),
    ('+ RANSAC only',          True,  False),
    ('+ AQE only',             False, True),
    ('+ RANSAC + AQE (Full)',  True,  True),
]:
    mAP_v, _, _ = evaluate(eval_qs, ransac, aqe, label)
    ablation[label] = mAP_v

print(f'\nAblation — {eval_tag}')
for k, v in ablation.items():
    print(f'  {k:38s}: {v:.4f}')

---
# 📊 VOCABULARY SIZE EXPERIMENT

In [ ]:
vocab_map   = {}
# Use at most 20 queries per experiment for speed
eval_qs_sub = (eval_qs)[:20]

for k in [64, 128, 256, 512]:
    v_c = CACHE_DIR / f'vocab_K{k}.pkl'
    d_c = CACHE_DIR / f'vlad_raw_K{k}.npy'
    s_c = CACHE_DIR / f'db_stems_K{k}.pkl'
    p_c = CACHE_DIR / f'pca_K{k}_D{PCA_DIM}.pkl'

    vocab_k  = build_vocabulary(ALL_IMG_PATHS, k=k,
                                 n_sample_imgs=min(1500, N_IMAGES),
                                 cache_path=v_c)
    cents_k  = vocab_k.cluster_centers_.astype(np.float32)
    vlad_k, stems_k = build_database(ALL_IMG_PATHS, cents_k,
                                      cache_vlad=d_c, cache_stems=s_c)
    pca_k    = fit_pca_whitening(vlad_k, n_components=min(PCA_DIM, k*128),
                                  cache_path=p_c)
    db_k     = apply_pca_whitening(vlad_k, pca_k)

    # Temporarily swap global state
    _bak = (DB_VLAD.copy(), list(DB_STEMS), dict(STEM_TO_IDX),
            CENTROIDS.copy(), PCA_MODEL)

    DB_VLAD[:] = db_k if db_k.shape==DB_VLAD.shape else db_k
    # Re-assign module-level names so run_query uses new data
    globals().update(dict(
        DB_VLAD=db_k, DB_STEMS=stems_k,
        STEM_TO_IDX={s: i for i, s in enumerate(stems_k)},
        CENTROIDS=cents_k, PCA_MODEL=pca_k))

    mAP_k, _, _ = evaluate(eval_qs_sub, False, False, f'K={k}')
    vocab_map[k] = mAP_k

    # Restore
    _db, _stems, _s2i, _cents, _pca = _bak
    globals().update(dict(
        DB_VLAD=_db, DB_STEMS=_stems, STEM_TO_IDX=_s2i,
        CENTROIDS=_cents, PCA_MODEL=_pca))

print('\nVocabulary experiment:', vocab_map)

---
# 🖼️ QUALITATIVE EVALUATION — Top-5 Results

In [ ]:
def show_top5(query_info: dict, ranked_stems: list, n_show: int = 5):
    """
    Display query image + top-5 retrieved results.
    Green border = relevant, Red = irrelevant, Yellow box = query ROI.
    """
    pos  = query_info['good'] | query_info['ok']
    junk = query_info['junk']
    top5 = [s for s in ranked_stems if s not in junk][:n_show]

    fig, axes = plt.subplots(1, n_show+1, figsize=(4*(n_show+1), 4))
    qname = query_info['name'].replace('oxford_','').replace('paris_','')

    def load(stem):
        p = STEM2PATH.get(stem)
        if p is None: return np.zeros((200,200,3),dtype=np.uint8)
        img = cv2.imread(str(p))
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None \
               else np.zeros((200,200,3),dtype=np.uint8)

    ax = axes[0]
    ax.imshow(load(query_info['query_img']))
    if query_info['query_roi']:
        x1,y1,x2,y2 = query_info['query_roi']
        ax.add_patch(patches.Rectangle(
            (x1,y1),x2-x1,y2-y1,lw=3,edgecolor='yellow',facecolor='none'))
    ax.set_title(f'QUERY\n{qname}', fontsize=9, fontweight='bold')
    ax.axis('off')

    for i, stem in enumerate(top5):
        ax = axes[i+1]
        ax.imshow(load(stem))
        rel   = stem in pos
        color = 'limegreen' if rel else 'tomato'
        ax.set_title(f'#{i+1} {"✓" if rel else "✗"}',
                     color=color, fontsize=11, fontweight='bold')
        for sp in ax.spines.values():
            sp.set_edgecolor(color); sp.set_linewidth(5)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'query_{qname}.png', dpi=90, bbox_inches='tight')
    plt.show()


res_main  = res_ox_f if oxford_queries else res_pa_f
qs_main   = oxford_queries if oxford_queries else paris_queries
ds_label  = 'Oxford' if oxford_queries else 'Paris'

sorted_qs = sorted(
    [q for q in qs_main if q['name'] in res_main],
    key=lambda q: res_main[q['name']]['ap'], reverse=True)

for label, q in [('BEST',   sorted_qs[0]),
                  ('MEDIAN', sorted_qs[len(sorted_qs)//2]),
                  ('WORST',  sorted_qs[-1])]:
    r = res_main[q['name']]
    print(f'\n── {label} query — AP={r["ap"]:.3f} ──')
    show_top5(q, r['ranked'])

---
# 📊 RESULTS DASHBOARD

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Image Retrieval — Results Dashboard', fontsize=17, fontweight='bold')

# 1. Per-query AP
ax = axes[0,0]
if res_main:
    names  = [k.replace('oxford_','').replace('paris_','') for k in res_main]
    aps    = [res_main[k]['ap'] for k in res_main]
    med    = float(np.median(aps))
    colors = ['steelblue' if a>=med else 'tomato' for a in aps]
    ax.barh(range(len(names)), aps, color=colors, edgecolor='white', height=0.7)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
    ax.axvline(np.mean(aps), color='k', ls='--', lw=1.5,
               label=f'mAP={np.mean(aps):.3f}')
    ax.legend(); ax.set_xlabel('AP'); ax.set_title(f'{ds_label} Per-Query AP')
    ax.set_xlim(0,1)

# 2. Ablation
ax = axes[0,1]
if ablation:
    labs  = list(ablation.keys())
    vals  = list(ablation.values())
    bars  = ax.barh(labs, vals,
                    color=['#4C72B0','#DD8452','#55A868','#C44E52'],
                    edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(v+0.005, bar.get_y()+bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=9)
    ax.set_xlabel('mAP'); ax.set_title(f'{eval_tag} Ablation Study')
    ax.set_xlim(0, max(vals)*1.15)

# 3. Vocabulary size
ax = axes[1,0]
if vocab_map:
    ks, ms = list(vocab_map.keys()), list(vocab_map.values())
    ax.plot(ks, ms, 'o-', color='steelblue', lw=2, ms=9)
    for k, m in zip(ks, ms):
        ax.annotate(f'{m:.3f}', (k,m), xytext=(0,10),
                    textcoords='offset points', ha='center', fontsize=9)
    ax.set_xlabel('K (vocabulary size)'); ax.set_ylabel('mAP')
    ax.set_title('Vocabulary Size vs mAP')
    ax.set_xscale('log', base=2)
    ax.set_xticks(ks); ax.set_xticklabels([str(k) for k in ks])
    ax.grid(True, alpha=0.3)

# 4. AP histogram
ax = axes[1,1]
if res_main:
    all_ap = [r['ap'] for r in res_main.values()]
    ax.hist(all_ap, bins=12, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(all_ap),   color='red',    ls='--', lw=1.5,
               label=f'Mean={np.mean(all_ap):.3f}')
    ax.axvline(np.median(all_ap), color='orange', ls=':',  lw=1.5,
               label=f'Median={np.median(all_ap):.3f}')
    ax.set_xlabel('AP'); ax.set_ylabel('# Queries')
    ax.set_title('AP Distribution (Full Pipeline)'); ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'results_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Dashboard saved.')

---
# 📋 FINAL SUMMARY

In [ ]:
W = 62
def row(label, val, W=W):
    content = f'  {label}: {val}'
    print(f'║{content:<{W-2}}║')

print('╔' + '═'*(W-2) + '╗')
print('║' + ' HANDCRAFTED IMAGE RETRIEVAL — SUMMARY'.center(W-2) + '║')
print('╠' + '═'*(W-2) + '╣')
row('DB images', N_IMAGES)
row('Oxford queries', len(oxford_queries))
row('Paris  queries', len(paris_queries))
row('K_VLAD', K_VLAD)
row('PCA dim', PCA_DIM)
print('╠' + '═'*(W-2) + '╣')
if oxford_queries:
    row('Oxford baseline mAP', f'{mAP_ox_b:.4f}')
    row('Oxford full     mAP', f'{mAP_ox_f:.4f}  (+{(mAP_ox_f-mAP_ox_b)*100:.1f}%)')
if paris_queries:
    row('Paris  baseline mAP', f'{mAP_pa_b:.4f}')
    row('Paris  full     mAP', f'{mAP_pa_f:.4f}  (+{(mAP_pa_f-mAP_pa_b)*100:.1f}%)')
print('╠' + '═'*(W-2) + '╣')
for label, val in ablation.items():
    row(label[:40], f'{val:.4f}')
if vocab_map:
    print('╠' + '═'*(W-2) + '╣')
    best_k = max(vocab_map, key=vocab_map.get)
    row(f'Best K', f'K={best_k}  mAP={vocab_map[best_k]:.4f}')
print('╚' + '═'*(W-2) + '╝')